# 2022 NHTS dataset inspection

**Scope:** structural inspection only. This notebook reads the four supplied day-travel tables without modifying `data/raw/`. It does not clean data, create a processed dataset, plot results, or implement any EV/grid/ML model.

**Official references:** `data/raw/nhts_2022/docs/user_guide.pdf` and `codebook.pdf` (Public Use Codebook v2.0.1, December 2024). The accompanying variable map is `docs/methodology/nhts_variable_mapping.md`.


## File inventory and units of observation

The supplied files are `hhv2pub.csv` (Household), `perv2pub.csv` (Person), `vehv2pub.csv` (Vehicle), `tripv2pub.csv` (Trip), plus `ldtv2pub.csv` (Long Distance). The long-distance file is outside this inspection’s four-table scope.

The User Guide states that the Trip file contains a **person trip** for each household member who took a trip: if four family members travel together, this is four trip records. Hence a Trip row must not be treated as a unique vehicle movement.


In [1]:
from pathlib import Path

import pandas as pd

project_root = Path.cwd().resolve()
if not (project_root / "data").exists():
    project_root = project_root.parent

raw_dir = project_root / "data" / "raw" / "nhts_2022"
files = {
    "household": "hhv2pub.csv",
    "person": "perv2pub.csv",
    "vehicle": "vehv2pub.csv",
    "trip": "tripv2pub.csv",
}

# String loading preserves leading zeroes in IDs/codes and negative coded responses.
tables = {
    name: pd.read_csv(raw_dir / filename, dtype="string", keep_default_na=False)
    for name, filename in files.items()
}
household, person, vehicle, trip = (tables[name] for name in files)


## Dimensions and table grain

Observed dimensions: Household **7,893 × 43**; Person **16,997 × 153**; Vehicle **14,684 × 55**; Trip **31,074 × 102**.

- Household: one row per sampled household (`HOUSEID`).
- Person: one row per household member (`HOUSEID`, `PERSONID`).
- Vehicle: one row per household-roster vehicle (`VEHCASEID`; also `HOUSEID`, `VEHID`).
- Trip: one person-trip record (`TDCASEID`; also `HOUSEID`, `PERSONID`, `SEQ_TRIPID`).


In [2]:
summary = pd.DataFrame(
    {
        "rows": {name: len(frame) for name, frame in tables.items()},
        "columns": {name: frame.shape[1] for name, frame in tables.items()},
        "key_candidate": {
            "household": "HOUSEID",
            "person": "HOUSEID + PERSONID",
            "vehicle": "VEHCASEID (also HOUSEID + VEHID)",
            "trip": "TDCASEID (also HOUSEID + PERSONID + SEQ_TRIPID)",
        },
    }
).rename_axis("table")
summary


,rows,columns,key_candidate
table,,,
household,7893,43,HOUSEID
person,16997,153,HOUSEID + PERSONID
vehicle,14684,55,VEHCASEID (also HOUSEID + VEHID)
trip,31074,102,TDCASEID (also HOUSEID + PERSONID + SEQ_TRIPID)


## Relationships and key checks

Expected links are Household `1→many` Person, Household `1→many` Vehicle, and Person `1→many` Trip. Applicable Trip records link to Vehicle through `VEHCASEID`.

Inspection result: the stated primary/composite keys were unique, and no Person, Vehicle, or Trip row had an unmatched Household; no Trip row had an unmatched Person; all populated trip `VEHCASEID` values matched the Vehicle file. These are structural checks, not a decision to filter records.


In [3]:
person_key = ["HOUSEID", "PERSONID"]
vehicle_key = ["HOUSEID", "VEHID"]
trip_sequence_key = ["HOUSEID", "PERSONID", "SEQ_TRIPID"]

key_checks = pd.Series(
    {
        "HOUSEID unique in Household": household["HOUSEID"].is_unique,
        "HOUSEID + PERSONID unique in Person": not person.duplicated(person_key).any(),
        "VEHCASEID unique in Vehicle": vehicle["VEHCASEID"].is_unique,
        "HOUSEID + VEHID unique in Vehicle": not vehicle.duplicated(vehicle_key).any(),
        "TDCASEID unique in Trip": trip["TDCASEID"].is_unique,
        "person sequence key unique in Trip": not trip.duplicated(trip_sequence_key).any(),
        "Trip HOUSEID missing from Household": (~trip["HOUSEID"].isin(household["HOUSEID"])).sum(),
        "Trip person key missing from Person": (~trip.set_index(person_key).index.isin(person.set_index(person_key).index)).sum(),
        "Populated trip VEHCASEID missing from Vehicle": (
            ~trip.loc[trip["VEHCASEID"] != "-1", "VEHCASEID"].isin(vehicle["VEHCASEID"])
        ).sum(),
    },
    name="result",
)
key_checks


HOUSEID unique in Household                      True
HOUSEID + PERSONID unique in Person              True
VEHCASEID unique in Vehicle                      True
HOUSEID + VEHID unique in Vehicle                True
TDCASEID unique in Trip                          True
person sequence key unique in Trip               True
Trip HOUSEID missing from Household                 0
Trip person key missing from Person                 0
Populated trip VEHCASEID missing from Vehicle       0
Name: result, dtype: object

## Variables relevant to mobility behaviour

The detailed source-backed mapping is in `nhts_variable_mapping.md`. Core trip fields for later inspection are:

- IDs: `HOUSEID`, `PERSONID`, `TDCASEID`, `SEQ_TRIPID`, `VEHCASEID`, `VEHID`.
- Vehicle-use and occupant roles: `TRPHHVEH`, `DRVR_FLG`, `PSGR_FLG`, `WHODROVE`, `WHODROVE_IMP`.
- Time and distance: `STRTTIME`, `ENDTIME` (local 24-hour `HHMM`), `TRVLCMIN` (minutes), `TRPMILES` (calculated miles), and `DWELTIME` (destination minutes).
- Place/purpose and mode: `WHYFROM`, `WHYTO`, `WHYTRP1S`, `TRIPPURP`, `TRPTRANS`.
- Day context and weights: `TRAVDAY`, `TDAYDATE` (officially `YYYYMM`, not a full date), `TDWKND`, `WTTRDFIN` and its 2-/5-day variants.
- Vehicle characteristics: `VEHFUEL`, `VEHTYPE`, `VEHYEAR`, `MAKE`, `HYBRID`, `ANNMILES`, `VEHOWNED`, `WHOMAIN`.


In [4]:
trip_fields = [
    "TRPHHVEH", "VEHCASEID", "VEHID", "DRVR_FLG", "PSGR_FLG",
    "STRTTIME", "ENDTIME", "TRVLCMIN", "TRPMILES", "DWELTIME",
    "WHYFROM", "WHYTO", "WHYTRP1S", "TRIPPURP", "TRPTRANS",
    "TRAVDAY", "TDAYDATE", "TDWKND", "WTTRDFIN",
]

trip[trip_fields].head()


,TRPHHVEH,VEHCASEID,VEHID,DRVR_FLG,PSGR_FLG,STRTTIME,ENDTIME,TRVLCMIN,TRPMILES,DWELTIME,WHYFROM,WHYTO,WHYTRP1S,TRIPPURP,TRPTRANS,TRAVDAY,TDAYDATE,TDWKND,WTTRDFIN
0,01,900001300201,01,01,02,1435,1450,15,3.90242386575513,75,01,15,50,03,03,01,202202,01,1608361.95907888
1,01,900001300201,01,01,02,1605,1615,10,3.90242386575513,-9,15,01,01,03,03,01,202202,01,1608361.95907888
2,01,900001300202,02,01,02,0700,0730,30,17.0770665009323,10,01,12,80,02,03,01,202202,01,2208973.05782135
3,01,900001300202,02,01,02,0740,0750,10,4.74829086389061,10,12,12,80,05,03,01,202202,01,2208973.05782135
4,01,900001300202,02,01,02,0800,0830,30,14.0988191423244,-9,12,01,01,02,03,01,202202,01,2208973.05782135


## Special coded values and documentation issue

Negative values are valid survey codes, not automatically missing. The codebook commonly uses `-1` for appropriate/valid skip, `-7` for refusal, `-8` for don’t know, and `-9` for not ascertained; their meaning is variable-specific.

Observed examples: `STRTTIME`, `ENDTIME`, and `TRVLCMIN` each have 21 `-9` records; `TRPMILES` has 14 `-9` records; `DWELTIME` has 10,615 `-9` records; `VEHCASEID` has 4,767 `-1` records. No rows are discarded here.

The raw Trip CSV includes `TRIPMODE`, but the supplied official Codebook v2.0.1 does not document that field. The documented mode field is `TRPTRANS`; the notebook therefore does not interpret `TRIPMODE` codes.


In [5]:
special_codes = ["", "-1", "-7", "-8", "-9"]
special_code_counts = (
    trip[trip_fields]
    .isin(special_codes)
    .sum()
    .rename("count_of_blank_or_negative_special_codes")
    .sort_values(ascending=False)
)
special_code_counts


DWELTIME     10615
VEHID         4767
VEHCASEID     4767
PSGR_FLG      3665
DRVR_FLG      3665
TRPHHVEH      1168
WHYFROM         47
TRIPPURP        24
ENDTIME         21
TRVLCMIN        21
STRTTIME        21
TRPMILES        14
WHYTO            0
WHYTRP1S         0
TRPTRANS         0
TRAVDAY          0
TDAYDATE         0
TDWKND           0
WTTRDFIN         0
Name: count_of_blank_or_negative_special_codes, dtype: int64

## Candidate vehicle-level daily trip-chain reconstruction

A vehicle-level chain is **feasible with qualifications**. Use only a candidate scope first: `TRPHHVEH == "01"`, a populated `VEHCASEID`, and `DRVR_FLG == "01"`; then order by local `STRTTIME` within `VEHCASEID`. In the inspected data this yields 20,067 driver person-trip records for 6,943 identified household vehicles.

This is not yet an aggregation rule. The raw person-trip scope contains 26,294 household-vehicle person trips. Grouping those records by vehicle, start/end clock times, and distance shows 5,430 rows beyond one representative per same-time/distance group, consistent with the User Guide’s person-trip definition. Even the candidate driver scope has 57 such additional rows, so it requires case review before any deduplication. Also retain 6 candidate-driver records with missing start/end and 31 whose end clock time precedes start time; an exact calendar day is unavailable in the public `TDAYDATE` field.


In [6]:
household_vehicle_person_trips = trip.loc[
    (trip["TRPHHVEH"] == "01") & (trip["VEHCASEID"] != "-1")
]
candidate_driver_trips = household_vehicle_person_trips.loc[
    household_vehicle_person_trips["DRVR_FLG"] == "01"
]

# Diagnostic only: do not deduplicate or write a derived dataset in this notebook.
movement_proxy = ["VEHCASEID", "STRTTIME", "ENDTIME", "TRPMILES"]
same_time_distance = candidate_driver_trips.groupby(movement_proxy, dropna=False).size()

pd.Series(
    {
        "household-vehicle person-trip rows": len(household_vehicle_person_trips),
        "candidate driver rows": len(candidate_driver_trips),
        "identified vehicles in candidate driver scope": candidate_driver_trips["VEHCASEID"].nunique(),
        "candidate rows beyond one per same-time/distance group": len(candidate_driver_trips) - len(same_time_distance),
        "candidate records with -9 start or end": (
            (candidate_driver_trips["STRTTIME"] == "-9")
            | (candidate_driver_trips["ENDTIME"] == "-9")
        ).sum(),
        "candidate records with end clock time before start": (
            (candidate_driver_trips["STRTTIME"] != "-9")
            & (candidate_driver_trips["ENDTIME"] != "-9")
            & (candidate_driver_trips["ENDTIME"] < candidate_driver_trips["STRTTIME"])
        ).sum(),
    },
    name="inspection result",
)


household-vehicle person-trip rows                        26294
candidate driver rows                                     20067
identified vehicles in candidate driver scope              6943
candidate rows beyond one per same-time/distance group       57
candidate records with -9 start or end                        6
candidate records with end clock time before start           31
Name: inspection result, dtype: int64

## Inspection conclusion

The immediate next step should be a **non-destructive travel EDA plan** for documented trip times, distance, purposes, and vehicle-use/driver-status fields. Before producing vehicle-day summaries or trip-chain examples, inspect the same-vehicle/same-time candidate records and define a documented rule for identifying one vehicle movement without erasing legitimate trips.
